In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [29]:

np.random.seed(42)

pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## Part 1 - Data cleanning & Preprocessing 

In [30]:
dataset_a = pd.DataFrame({
    "student_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "age": [20, 21, np.nan, 22, 23, 20, np.nan, 24],
    "study_hours": [10, 12, 8, np.nan, 15, 10, 9, 14],
    "score": [85, 90, 78, 88, np.nan, 82, 80, 91]
})

dataset_a.head()

,student_id,age,study_hours,score
0,1,20.000,10.000,85.000
1,2,21.000,12.000,90.000
2,3,NaN,8.000,78.000
3,4,22.000,NaN,88.000
4,5,23.000,15.000,NaN


In [31]:
print("Missing values per column:")
print(dataset_a.isnull().sum())

Missing values per column:
student_id     0
age            2
study_hours    1
score          1
dtype: int64


In [32]:
dataset_a_clean = dataset_a.copy()

for col in ["age", "study_hours", "score"]:
    mean_value = dataset_a_clean[col].mean()
    dataset_a_clean[col] = dataset_a_clean[col].fillna(mean_value)

dataset_a_clean

,student_id,age,study_hours,score
0,1,20.000,10.000,85.000
1,2,21.000,12.000,90.000
2,3,21.667,8.000,78.000
3,4,22.000,11.143,88.000
4,5,23.000,15.000,84.857
5,6,20.000,10.000,82.000
6,7,21.667,9.000,80.000
7,8,24.000,14.000,91.000


**Note** The mean is only one of several imputation strategies. Other common options are the **median** (more robust to outliers), the **mode** (useful for categorical columns), or more advanced techniques like KNN imputation. The right choice depends on the data.

In [33]:
dataset_b = pd.DataFrame({
    "brand": ["Toyota", "toyota", "HONDA", "Ford", "ford", "Toyota", "Honda", "TOYOTA"],
    "model": ["Corolla", "Corolla", "Civic", "Focus", "Focus", "Corolla", "Civic", "Corolla"],
    "price": ["15000", "15000", "18000", "12000", "15000", "18500", "15000", "15000"],
    "year": [2018, 2018, 2019, 2017, 2017, 2018, 2020, 2018],
})

In [34]:
#Step 1: check data types.
#Notice that price is an object (string) instead of numeric.
dataset_b.dtypes

brand      str
model      str
price      str
year     int64
dtype: object

In [35]:
#Step 2: normalize text in the brand column
# -strip() removes leading and trailing whitespace
# -title() converts to title case (first letter capitalized)

dataset_b_clean = dataset_b.copy()
dataset_b_clean["brand"] = dataset_b_clean["brand"].str.strip().str.title()    

In [36]:
#Step 3: convert price from string to numeric
dataset_b_clean["price"] = pd.to_numeric(dataset_b_clean["price"])

In [37]:
##Step 4: remove duplicates rows
print (f"Rows before removing duplicates: {len(dataset_b_clean)}")
dataset_b_clean = dataset_b_clean.drop_duplicates().reset_index(drop=True)
print(f"Rows after removing duplicates: {len(dataset_b_clean)}")

dataset_b_clean

Rows before removing duplicates: 8
Rows after removing duplicates: 6


,brand,model,price,year
0,Toyota,Corolla,15000,2018
1,Honda,Civic,18000,2019
2,Ford,Focus,12000,2017
3,Ford,Focus,15000,2017
4,Toyota,Corolla,18500,2018
5,Honda,Civic,15000,2020


In [38]:
#Verify data types after cleaning
dataset_b_clean.dtypes

brand      str
model      str
price    int64
year     int64
dtype: object

## Dataset C - Outliers and Feature Scaling
Outliers are values that are very different from the rest of the data.
They can be legitimate, but they can also be errors (eg. a typo).

Feature sealing is important when features live on very different numerical ranges - for example, a car's weight (thousands of pounds) vs.
Its number of cylinders (4-0), Without sealing, features with larger ranges can dominate the learning process.
markdot

In [40]:
dataset_c = pd.DataFrame({
    "weight": [3.50,3.69,3.44,3.43,4.34,4.42,2.37,25.00],
    "horsepower":[130,165,150,140,198,220,95,180],
    "cylinders": [4, 6, 4, 4, 8, 8, 4, 6],
    "mpg":[18,15,18,16,15,14,24,17]

})

dataset_c

,weight,horsepower,cylinders,mpg
0,3.500,130,4,18
1,3.690,165,6,15
2,3.440,150,4,18
3,3.430,140,4,16
4,4.340,198,8,15
5,4.420,220,8,14
6,2.370,95,4,24
7,25.000,180,6,17


In [43]:
#Step 1: detect outliers in 'weight' using the IQR (InterQuartile Range) rule.
# A value is considered an outlier if it falls outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR].
q1 = dataset_c["weight"].quantile(0.25)
q3 = dataset_c["weight"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}")
print(f"Valid range for 'weight': [{lower_bound:.2f}, {upper_bound:.2f}]")

outliers_mask = (dataset_c["weight"] < lower_bound) | (dataset_c["weight"] > upper_bound)
print("\nOutlier rows:")
print(dataset_c[outliers_mask])

Q1 = 3.44, Q3 = 4.36, IQR = 0.92
Valid range for 'weight': [2.05, 5.74]

Outlier rows:
   weight  horsepower  cylinders  mpg
7  25.000         180          6   17
